# registry

> Discover skills from `pyskills` entry points and `SKILL.md` directories, and load either kind as
> model context through one interface.

In [ ]:
#| default_exp registry

In [ ]:
#| hide
from nbdev.showdoc import *

A skill is either a **pyskill** -- a Python module registered under the `pyskills` entry-point
group, discovered without importing (its description is the first paragraph of the module docstring,
read via AST) -- or a **SKILL.md** -- a markdown file with YAML frontmatter, the format Claude Code
and similar harnesses use. `Registry` merges both into one list the finder can rank and the harness
can load.

In [ ]:
#| export
import re, importlib
from dataclasses import dataclass, field
from pathlib import Path
from importlib.metadata import entry_points
from fastcore.utils import store_attr
from pyskills.core import ep_desc, doc

In [ ]:
#| export
@dataclass
class Skill:
    'A unit of agent capability: a pyskills module (kind="py") or a SKILL.md file (kind="md").'
    name:str                                    # short name shown in the catalog
    kind:str                                    # 'py' or 'md'
    description:str                             # discovery description, used by the finder
    source:str                                  # module path (py) or SKILL.md path (md)
    triggers:list = field(default_factory=list) # trigger phrases from SKILL.md frontmatter
    def one_line(self): return ' '.join(self.description.split())

In [ ]:
#| export
def parse_frontmatter(txt):
    'Split a SKILL.md into (frontmatter dict, body). Tolerates missing or unparsable frontmatter.'
    m = re.match(r'^---\s*\n(.*?)\n---\s*\n?(.*)$', txt, re.S)
    if not m: return {}, txt
    try:
        import yaml
        meta = yaml.safe_load(m.group(1)) or {}
        if not isinstance(meta, dict): meta = {}
    except Exception: meta = {}
    return meta, m.group(2)

In [ ]:
fm, body = parse_frontmatter('---\nname: x\ndescription: does x\ntriggers:\n  - about to x\n---\n# X\nbody here')
assert fm['name']=='x' and fm['triggers']==['about to x'] and body.startswith('# X')
assert parse_frontmatter('no frontmatter') == ({}, 'no frontmatter')

In [ ]:
#| export
def pyskill_skills():
    'One `Skill` per `pyskills` entry point that has a readable description (no imports happen).'
    res = []
    for ep in entry_points().select(group='pyskills'):
        try: d = ep_desc(ep)   # ep_desc ast-parses the module; a syntax error there must not kill discovery
        except Exception: continue
        if d: res.append(Skill(name=ep.name, kind='py', description=d, source=ep.value))
    return res

In [ ]:
#| export
def skillmd_skills(root=None, dirs=()):
    'One `Skill` per `*/SKILL.md` under `root`\'s `.claude/skills` and `.agents/skills`, plus extra `dirs`.'
    root = Path(root or '.')
    res = []
    for d in [root/'.claude/skills', root/'.agents/skills', *map(Path, dirs)]:
        if not d.exists(): continue
        for p in sorted(d.glob('*/SKILL.md')):
            meta, _ = parse_frontmatter(p.read_text())
            trigs = [str(t) for t in (meta.get('triggers') or [])]
            res.append(Skill(name=str(meta.get('name', p.parent.name)), kind='md',
                             description=str(meta.get('description','')).strip(), source=str(p), triggers=trigs))
    return res

In [ ]:
#| export
class Registry:
    'Unified skill registry over pyskills entry points and SKILL.md directories.'
    def __init__(self, root=None, dirs=(), include_py=True, include_md=True):
        store_attr()
        self.refresh()
    def refresh(self):
        'Re-discover skills; pyskills win name collisions with SKILL.md copies of the same skill.'
        sks = (pyskill_skills() if self.include_py else []) + (skillmd_skills(self.root, self.dirs) if self.include_md else [])
        seen, self.skills = set(), []
        for s in sks:
            if s.name in seen: continue
            seen.add(s.name); self.skills.append(s)
        return self
    def __len__(self): return len(self.skills)
    def __iter__(self): return iter(self.skills)
    def get(self, name): return next((s for s in self.skills if s.name==name), None)
    def catalog(self):
        'One `- name: description` line per skill, for the system prompt.'
        return '\n'.join(f'- {s.name}: {s.one_line()}' for s in self.skills)
    def load(self, skill):
        'Context text for `skill` (a `Skill` or name): the SKILL.md body, or the pyskill\'s rendered `doc()`.'
        s = self.get(skill) if isinstance(skill, str) else skill
        if s is None: raise KeyError(skill)
        if s.kind=='md': return parse_frontmatter(Path(s.source).read_text())[1].strip()
        mod = importlib.import_module(s.source)
        return f'Loaded pyskill `{s.source}` -- call it from python (`from {s.source} import *`).\n\n{doc(mod)}'

In [ ]:
import tempfile, os
tmp = tempfile.mkdtemp()
d = Path(tmp)/'.claude/skills/demo'; d.mkdir(parents=True)
(d/'SKILL.md').write_text('---\nname: demo\ndescription: a demo skill\ntriggers:\n  - demo me\n---\nUse `demo()` to demo.')
r = Registry(root=tmp, include_py=False)
assert [s.name for s in r] == ['demo'] and r.get('demo').triggers == ['demo me']
assert r.catalog() == '- demo: a demo skill'
assert r.load('demo') == 'Use `demo()` to demo.'

In [ ]:
# pyskills discovery runs against whatever this env has installed; it must not blow up
ps = pyskill_skills()
assert all(s.kind=='py' and s.description for s in ps)
# duplicate names dedup: a SKILL.md named like an existing pyskill is dropped
if ps:
    dup = Path(tmp)/'.claude/skills'/ps[0].name; dup.mkdir(parents=True, exist_ok=True)
    (dup/'SKILL.md').write_text(f'---\nname: {ps[0].name}\ndescription: shadow\n---\nshadow')
    r2 = Registry(root=tmp)
    assert r2.get(ps[0].name).kind == 'py'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()